In [1]:
!pip install -q ucimlrepo fastapi uvicorn scikit-learn pandas numpy joblib pytest httpx

In [2]:
import os
import json
import joblib
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
student_data = fetch_ucirepo(id=320)

X = student_data.data.features.copy()
y = student_data.data.targets.iloc[:, 0].copy()

print("Dataset loaded successfully!")

print("\nFirst five records:")
display(X.head())

print("\nDataset shape:", X.shape)

Dataset loaded successfully!

First five records:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,yes,no,no,4,3,4,1,1,3,4
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,yes,yes,no,5,3,3,1,1,3,2
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,yes,yes,no,4,3,2,2,3,3,6
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,yes,yes,3,2,2,1,1,5,0
4,GP,F,16,U,GT3,T,3,3,other,other,...,yes,no,no,4,3,2,1,2,5,0



Dataset shape: (649, 30)


In [4]:
print("Dataset information:")
X.info()

print("\nStatistical summary:")
display(X.describe(include="all"))

print("\nMissing values:")
print(X.isnull().sum())

print("\nDuplicate records:", X.duplicated().sum())

# Remove earlier period grades to avoid target leakage
X = X.drop(columns=["G1", "G2"], errors="ignore")

# Convert final grade into numeric
y = pd.to_numeric(y, errors="coerce")

# Remove rows with missing target
valid_rows = y.notna()

X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

# Convert final grade into pass/fail classification
y = (y >= 10).astype(int)

print("\nTarget distribution:")
print(y.value_counts())

print("\n0 = Pass")
print("1 = Fail")

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 30 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      649 non-null    object
 1   sex         649 non-null    object
 2   age         649 non-null    int64 
 3   address     649 non-null    object
 4   famsize     649 non-null    object
 5   Pstatus     649 non-null    object
 6   Medu        649 non-null    int64 
 7   Fedu        649 non-null    int64 
 8   Mjob        649 non-null    object
 9   Fjob        649 non-null    object
 10  reason      649 non-null    object
 11  guardian    649 non-null    object
 12  traveltime  649 non-null    int64 
 13  studytime   649 non-null    int64 
 14  failures    649 non-null    int64 
 15  schoolsup   649 non-null    object
 16  famsup      649 non-null    object
 17  paid        649 non-null    object
 18  activities  649 non-null    object
 19  nursery     649 non-null    o

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
count,649,649,649.000000,649,649,649,649.000000,649.000000,649,649,...,649,649,649,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000
unique,2,2,NaN,2,2,2,NaN,NaN,5,5,...,2,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,GP,F,NaN,U,GT3,T,NaN,NaN,other,other,...,yes,yes,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,423,383,NaN,452,457,569,NaN,NaN,258,367,...,580,498,410,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,16.744222,NaN,NaN,NaN,2.514638,2.306626,NaN,NaN,...,NaN,NaN,NaN,3.930663,3.180277,3.184900,1.502311,2.280431,3.536210,3.659476
std,NaN,NaN,1.218138,NaN,NaN,NaN,1.134552,1.099931,NaN,NaN,...,NaN,NaN,NaN,0.955717,1.051093,1.175766,0.924834,1.284380,1.446259,4.640759
min,NaN,NaN,15.000000,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,...,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
25%,NaN,NaN,16.000000,NaN,NaN,NaN,2.000000,1.000000,NaN,NaN,...,NaN,NaN,NaN,4.000000,3.000000,2.000000,1.000000,1.000000,2.000000,0.000000
50%,NaN,NaN,17.000000,NaN,NaN,NaN,2.000000,2.000000,NaN,NaN,...,NaN,NaN,NaN,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,2.000000
75%,NaN,NaN,18.000000,NaN,NaN,NaN,4.000000,3.000000,NaN,NaN,...,NaN,NaN,NaN,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,6.000000



Missing values:
school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
dtype: int64

Duplicate records: 0

Target distribution:
G1
1    492
0    157
Name: count, dtype: int64

0 = Pass
1 = Fail


In [5]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']

Categorical features:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Data splitting and preprocessing completed!")
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Data splitting and preprocessing completed!
Training data: (519, 30)
Testing data: (130, 30)


In [7]:
models = {

    "Logistic Regression": (
        LogisticRegression(max_iter=2000),
        {
            "classifier__C": [0.1, 1, 10]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            "classifier__max_depth": [3, 5, None]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 5, 10]
        }
    ),

    "Support Vector Machine": (
        SVC(probability=True, random_state=42),
        {
            "classifier__C": [0.1, 1, 10],
            "classifier__kernel": ["linear", "rbf"]
        }
    )
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Four machine learning algorithms defined!")

Four machine learning algorithms defined!


In [8]:
results = []
best_models = {}

for name, (classifier, parameters) in models.items():

    print("\nTraining:", name)

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    grid = GridSearchCV(
        pipeline,
        parameters,
        scoring="f1",
        cv=cv,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_models[name] = best_model

    predictions = best_model.predict(X_test)
    probabilities = best_model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "CV F1": grid.best_score_,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, probabilities)
    })

    print("Best parameters:", grid.best_params_)

comparison = pd.DataFrame(results)

comparison = comparison.sort_values(
    by="CV F1",
    ascending=False
)

print("\nMODEL COMPARISON")
display(comparison)


Training: Logistic Regression
Best parameters: {'classifier__C': 0.1}

Training: Decision Tree
Best parameters: {'classifier__max_depth': 5}

Training: Random Forest
Best parameters: {'classifier__max_depth': 5, 'classifier__n_estimators': 100}

Training: Support Vector Machine
Best parameters: {'classifier__C': 0.1, 'classifier__kernel': 'linear'}

MODEL COMPARISON


,Model,CV F1,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.885581,0.838462,0.830508,0.989899,0.903226,0.800912
3,Support Vector Machine,0.882586,0.830769,0.823529,0.989899,0.899083,0.790811
2,Random Forest,0.878686,0.823077,0.811475,1.000000,0.895928,0.853372
1,Decision Tree,0.865088,0.815385,0.857143,0.909091,0.882353,0.729717


In [9]:
champion_name = comparison.iloc[0]["Model"]

champion_model = best_models[champion_name]

print("Champion Model:", champion_name)

predictions = champion_model.predict(X_test)

probabilities = champion_model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(
    y_test,
    predictions,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("\nROC AUC:")
print(roc_auc_score(y_test, probabilities))

Champion Model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.35      0.51        31
           1       0.83      0.99      0.90        99

    accuracy                           0.84       130
   macro avg       0.87      0.67      0.71       130
weighted avg       0.85      0.84      0.81       130


Confusion Matrix:
[[11 20]
 [ 1 98]]

ROC AUC:
0.8009123492994461


In [10]:
os.makedirs("model", exist_ok=True)

joblib.dump(
    champion_model,
    "model/champion_model.joblib"
)

print("Champion model saved successfully!")

Champion model saved successfully!


In [11]:
sample_input = X_test.iloc[0].to_dict()

# Convert values into JSON-compatible types
sample_input = {
    key: (
        value.item()
        if isinstance(value, np.generic)
        else value
    )
    for key, value in sample_input.items()
}

with open("sample_input.json", "w") as file:
    json.dump(sample_input, file, indent=4)

print("Sample API input created!")
print(json.dumps(sample_input, indent=4))

Sample API input created!
{
    "school": "GP",
    "sex": "F",
    "age": 17,
    "address": "U",
    "famsize": "GT3",
    "Pstatus": "T",
    "Medu": 4,
    "Fedu": 3,
    "Mjob": "other",
    "Fjob": "other",
    "reason": "reputation",
    "guardian": "mother",
    "traveltime": 1,
    "studytime": 2,
    "failures": 0,
    "schoolsup": "no",
    "famsup": "no",
    "paid": "no",
    "activities": "no",
    "nursery": "yes",
    "higher": "yes",
    "internet": "yes",
    "romantic": "yes",
    "famrel": 3,
    "freetime": 4,
    "goout": 5,
    "Dalc": 2,
    "Walc": 4,
    "health": 1,
    "absences": 16
}


In [12]:
%%writefile app.py

import joblib
import pandas as pd

from pathlib import Path
from typing import Dict, Any

from fastapi import FastAPI, HTTPException

app = FastAPI(
    title="Student Performance Prediction API",
    description="Machine Learning REST API for Student Pass/Fail Prediction",
    version="1.0.0"
)

MODEL_PATH = (
    Path(__file__).resolve().parent
    / "model"
    / "champion_model.joblib"
)

model = joblib.load(MODEL_PATH)

FEATURES = list(model.feature_names_in_)


@app.get("/")
def home():

    return {
        "message": "Student Performance Prediction API",
        "status": "API is running"
    }


@app.post("/predict")
def predict(data: Dict[str, Any]):

    missing = [
        feature for feature in FEATURES
        if feature not in data
    ]

    if missing:
        raise HTTPException(
            status_code=422,
            detail=f"Missing features: {missing}"
        )

    try:

        input_data = pd.DataFrame(
            [[data[feature] for feature in FEATURES]],
            columns=FEATURES
        )

        prediction = int(model.predict(input_data)[0])

        probabilities = model.predict_proba(input_data)[0]

        return {
            "prediction": prediction,
            "result": (
                "Fail" if prediction == 1 else "Pass"
            ),
            "probability": {
                "pass": float(probabilities[0]),
                "fail": float(probabilities[1])
            },
            "disclaimer": "Educational prediction only."
        }

    except Exception as error:

        raise HTTPException(
            status_code=400,
            detail=str(error)
        )

Writing app.py


In [13]:
%%writefile test_app.py

import json

from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

with open("sample_input.json") as file:
    sample_data = json.load(file)


def test_home():

    response = client.get("/")

    assert response.status_code == 200


def test_prediction():

    response = client.post(
        "/predict",
        json=sample_data
    )

    assert response.status_code == 200

    assert response.json()["prediction"] in [0, 1]


def test_missing_feature():

    data = sample_data.copy()

    data.pop(next(iter(data)))

    response = client.post(
        "/predict",
        json=data
    )

    assert response.status_code == 422

Writing test_app.py


In [14]:
!pytest -v test_app.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 3 items                                                              

test_app.py::test_home PASSED                                            [ 33%]
test_app.py::test_prediction PASSED                                      [ 66%]
test_app.py::test_missing_feature PASSED                                 [100%]

============================== 3 passed in 2.37s ===============================


In [15]:
%%writefile requirements.txt

ucimlrepo
fastapi
uvicorn
scikit-learn
pandas
numpy
joblib
pytest
httpx

Writing requirements.txt


In [16]:
%%writefile Dockerfile

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


In [17]:
%%writefile README.md

# Student Performance Analysis Using Machine Learning

## Project Overview

This project analyzes student academic and related information
and predicts whether a student is likely to pass or fail.

## Dataset

UCI Student Performance Dataset

https://archive.ics.uci.edu/dataset/320/student+performance

## Algorithms Used

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine

## Preprocessing

- Missing Value Handling
- Feature Scaling
- Categorical Encoding
- Train-Test Splitting

## Model Optimization

- GridSearchCV
- Stratified K-Fold Cross-Validation
- Champion Model Selection

## Evaluation

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Confusion Matrix

## API

FastAPI REST API

GET /
POST /predict

## Testing

Pytest

## Docker

Dockerfile included.

## Run API

uvicorn app:app --reload

## API Documentation

http://127.0.0.1:8000/docs

## Responsible Usage

This project is intended for educational purposes.
Predictions should not be used to label students or make
important academic decisions without human review.

Writing README.md


In [18]:
import subprocess
import time

api_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ]
)

time.sleep(5)

print("FastAPI server started!")

FastAPI server started!


In [19]:
from google.colab.output import eval_js

api_url = eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("API URL:", api_url)
print("Swagger Documentation:", api_url + "/docs")

API URL: https://8000-m-s-use1c5-3dw1d3vx183u9-c.us-east1.prod.r.colab.dev
Swagger Documentation: https://8000-m-s-use1c5-3dw1d3vx183u9-c.us-east1.prod.r.colab.dev/docs


In [20]:
import zipfile
from google.colab import files

project_files = [
    "app.py",
    "test_app.py",
    "requirements.txt",
    "Dockerfile",
    "README.md",
    "sample_input.json",
    "model/champion_model.joblib"
]

with zipfile.ZipFile(
    "student-performance-project.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for file in project_files:

        if os.path.exists(file):
            zip_file.write(file)

print("Project ZIP created successfully!")

files.download("student-performance-project.zip")

Project ZIP created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>